In [ ]:
import os
import io
import contextlib
from dotenv import load_dotenv

load_dotenv()

In [ ]:
GOOGLE_GEMINI_API_KEY = os.environ.get("GOOGLE_GEMINI_API_KEY")
assert GOOGLE_GEMINI_API_KEY is not None

In [ ]:
BRIGHTDATA_SERP_API_KEY = os.environ.get("BRIGHTDATA_SERP_API_KEY")
assert BRIGHTDATA_SERP_API_KEY is not None

In [ ]:
from langchain_brightdata import BrightDataSERP

serp_tool = BrightDataSERP(
    bright_data_api_key=BRIGHTDATA_SERP_API_KEY,
    search_engine="google",
    country="us",
    zone="serp_api1",
    parse_results=True
)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("gemini-3.5-flash", model_provider="google_genai", api_key=GOOGLE_GEMINI_API_KEY)

In [ ]:
def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

In [ ]:
from langchain.agents import create_agent


weather_agent = create_agent(
    model=model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant"
)

results = weather_agent.invoke(
    {"messages": [{"role":"user","content":"what is the weather in philly"}]}
)
results["messages"][-1].text

In [ ]:
search_agent = create_agent(
    model=model,
    tools=[serp_tool],
    system_prompt="You are a helpful assistant that can search the internet"
)
#Run the agent 
results = weather_agent.invoke(
    {"messages":[{"role":"user","content":"what is the weather in philly"}]}
)
results["messages"][-1].content

In [ ]:
for chunk in search_agent.stream(
    {"messages":[{"role":"user", "content":"Search for when Python was created"}]}
):
    # print(chunk.keys())
    agent_result = chunk.get("model")
    if agent_result:
        # print(agent_result)
        agent_msgs = agent_result.get("messages")
        if not agent_msgs:
            continue
        txt_response =  agent_msgs[-1].content
        if txt_response:
            print(txt_response)
    #results["message"][-1].content